# Evaluation: Post-hoc-Plausibilitätsvalidierung der Inferenzen

Dieses Notebook evaluiert die vom LLM extrahierten inferentiellen Relationen in zwei Schritten:

1. **Inter-Annotator Agreement (IAA):** Cohen’s Kappa für die Plausibilitätsurteile zweier Annotatoren
2. **LLM-Performance:** Accuracy, Precision, Recall und F1-Score gegen den Goldstandard (agreed annotations)

## Input

Eine XLSX-Datei mit **zwei Sheets** (eines pro Annotator). Jedes Sheet enthält die LLM-Annotationen
mit einer Spalte `Evaluation Inferenzen`, in der der Annotator jede Relation als `plausibel` oder
`nicht plausibel` bewertet hat.

Erwartete Spalten: Satz-ID, Originaltext, Propositions-ID, Proposition, Intra-Satz-Relationen,
Relationstypen, Evaluation Inferenzen

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    cohen_kappa_score, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)
from collections import Counter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ===== KONFIGURATION =====

INPUT_FILE = "Beutin_complete_INF+System_EVAL_V1.xlsx"              # <-- Hier anpassen

# Sheet-Namen: entweder Index (0, 1) oder Name ("Steffi", "Axel")
SHEET_ANNOTATOR_1 = 0                       # Erstes Sheet
SHEET_ANNOTATOR_2 = 1                       # Zweites Sheet

# Spaltennamen
COL_PROP_ID = "Propositions-ID"             # Unique Key pro Zeile
COL_RELTYP = "Relationstypen"               # Vom LLM annotierter Relationstyp
COL_EVAL = "Evaluation Inferenzen"          # Plausibilitätsurteil des Annotators
COL_INTRA_REL = "Intra-Satz-Relationen"     # Relationsdetails (für Fehleranalyse)

# Ausgabe
OUTPUT_FILE = Path(INPUT_FILE).stem + "_eval_ergebnisse.xlsx"

## 1. Daten laden und abgleichen

In [3]:
df1 = pd.read_excel(INPUT_FILE, sheet_name=SHEET_ANNOTATOR_1)
df2 = pd.read_excel(INPUT_FILE, sheet_name=SHEET_ANNOTATOR_2)

# Sheet-Namen ermitteln
xls = pd.ExcelFile(INPUT_FILE)
name1 = xls.sheet_names[SHEET_ANNOTATOR_1] if isinstance(SHEET_ANNOTATOR_1, int) else SHEET_ANNOTATOR_1
name2 = xls.sheet_names[SHEET_ANNOTATOR_2] if isinstance(SHEET_ANNOTATOR_2, int) else SHEET_ANNOTATOR_2

print(f"Annotator 1 ({name1}): {len(df1)} Zeilen")
print(f"Annotator 2 ({name2}): {len(df2)} Zeilen")
print(f"Spalten: {list(df1.columns)}")

# Spalten case-insensitiv finden
def find_col(df, target, name):
    if target in df.columns:
        return target
    matches = [c for c in df.columns if target.lower() in c.lower()]
    if matches:
        print(f"  Hinweis: '{target}' → '{matches[0]}' in {name}")
        return matches[0]
    print(f"  FEHLER: '{target}' nicht gefunden in {name}!")
    return None

for target in [COL_PROP_ID, COL_RELTYP, COL_EVAL, COL_INTRA_REL]:
    for label, df in [(name1, df1), (name2, df2)]:
        c = find_col(df, target, label)
        if c and c != target:
            df.rename(columns={c: target}, inplace=True)

Annotator 1 (Annotator_1): 106 Zeilen
Annotator 2 (Annotator_2): 106 Zeilen
Spalten: ['Kapitel', 'Satz-ID', 'Originaltext', 'Propositions-ID', 'Proposition', 'Intra-Satz-Relationen', 'Relationstypen', 'Evaluation Inferenzen', 'Anmerkungen Inferenzen', 'Reasoning', 'Systembezug']


In [4]:
# Abgleich über Propositions-ID
df1 = df1.set_index(COL_PROP_ID)
df2 = df2.set_index(COL_PROP_ID)

common = df1.index.intersection(df2.index)
only1 = df1.index.difference(df2.index)
only2 = df2.index.difference(df1.index)

print(f"\nGemeinsame Propositionen: {len(common)}")
if len(only1) > 0:
    print(f"  Nur in {name1}: {len(only1)}")
if len(only2) > 0:
    print(f"  Nur in {name2}: {len(only2)}")

df1 = df1.loc[common].copy()
df2 = df2.loc[common].copy()

# Evaluationsurteil normalisieren
def normalize_eval(val):
    if pd.isna(val):
        return None
    v = str(val).strip().lower()
    if v in ("plausibel", "ja", "yes", "j", "y", "1", "true", "p"):
        return "plausibel"
    if v in ("nicht plausibel", "nein", "no", "n", "0", "false", "np", "nicht"):
        return "nicht plausibel"
    print(f"  Warnung: Unbekannter Wert '{val}'")
    return None

df1["_eval"] = df1[COL_EVAL].apply(normalize_eval)
df2["_eval"] = df2[COL_EVAL].apply(normalize_eval)

# Zeilen ohne Urteil entfernen
valid = df1["_eval"].notna() & df2["_eval"].notna()
df1 = df1[valid]
df2 = df2[valid]

print(f"Gültige Urteile: {len(df1)}")
print(f"\nVerteilung {name1}: {Counter(df1['_eval'])}")
print(f"Verteilung {name2}: {Counter(df2['_eval'])}")


Gemeinsame Propositionen: 106
Gültige Urteile: 105

Verteilung Annotator_1: Counter({'plausibel': 88, 'nicht plausibel': 17})
Verteilung Annotator_2: Counter({'plausibel': 96, 'nicht plausibel': 9})


## 2. Inter-Annotator Agreement: Plausibilitätsurteil

In [5]:
y1 = df1["_eval"].values
y2 = df2["_eval"].values

kappa = cohen_kappa_score(y1, y2)
agreement = np.mean(y1 == y2)

labels = ["plausibel", "nicht plausibel"]
cm = confusion_matrix(y1, y2, labels=labels)
cm_df = pd.DataFrame(cm,
    index=[f"{name1}:{l}" for l in labels],
    columns=[f"{name2}:{l}" for l in labels])

def interpret_kappa(k):
    if k < 0:     return "schlecht (< 0)"
    if k < 0.21:  return "gering (0.00\u20130.20)"
    if k < 0.41:  return "ausreichend (0.21\u20130.40)"
    if k < 0.61:  return "moderat (0.41\u20130.60)"
    if k < 0.81:  return "substanziell (0.61\u20130.80)"
    return "fast perfekt (0.81\u20131.00)"

both_pl = ((y1 == "plausibel") & (y2 == "plausibel")).sum()
both_np = ((y1 == "nicht plausibel") & (y2 == "nicht plausibel")).sum()
disagree = (y1 != y2).sum()

print("=" * 60)
print("INTER-ANNOTATOR AGREEMENT: PLAUSIBILIT\u00c4TSURTEIL")
print("=" * 60)
print(f"Cohen's Kappa:       {kappa:.3f}")
print(f"%-\u00dcbereinstimmung:   {agreement*100:.1f}%")
print(f"Interpretation:      {interpret_kappa(kappa)}")
print(f"N:                   {len(y1)}")
print(f"\nKonfusionsmatrix:")
print(cm_df.to_string())
print(f"\nBeide 'plausibel':       {both_pl} ({both_pl/len(y1)*100:.1f}%)")
print(f"Beide 'nicht plausibel': {both_np} ({both_np/len(y1)*100:.1f}%)")
print(f"Disagreement:            {disagree} ({disagree/len(y1)*100:.1f}%)")

INTER-ANNOTATOR AGREEMENT: PLAUSIBILITÄTSURTEIL
Cohen's Kappa:       0.653
%-Übereinstimmung:   92.4%
Interpretation:      substanziell (0.61–0.80)
N:                   105

Konfusionsmatrix:
                             Annotator_2:plausibel  Annotator_2:nicht plausibel
Annotator_1:plausibel                           88                            0
Annotator_1:nicht plausibel                      8                            9

Beide 'plausibel':       88 (83.8%)
Beide 'nicht plausibel': 9 (8.6%)
Disagreement:            8 (7.6%)


## 3. Goldstandard konstruieren (Agreed Annotations)

In [6]:
agree_mask = df1["_eval"] == df2["_eval"]
df_agreed = df1[agree_mask].copy()
df_agreed["gold"] = df_agreed["_eval"]

df_disagree = df1[~agree_mask].copy()
df_disagree["A1_urteil"] = df1.loc[~agree_mask, "_eval"]
df_disagree["A2_urteil"] = df2.loc[~agree_mask, "_eval"]

print("=" * 60)
print("GOLDSTANDARD: AGREED ANNOTATIONS")
print("=" * 60)
print(f"Agreed Annotations:  {len(df_agreed)} von {len(df1)} ({len(df_agreed)/len(df1)*100:.1f}%)")
print(f"  davon 'plausibel':       {(df_agreed['gold'] == 'plausibel').sum()}")
print(f"  davon 'nicht plausibel': {(df_agreed['gold'] == 'nicht plausibel').sum()}")
print(f"Disagreements:       {len(df_disagree)} (aus Evaluation ausgeschlossen)")

GOLDSTANDARD: AGREED ANNOTATIONS
Agreed Annotations:  97 von 105 (92.4%)
  davon 'plausibel':       88
  davon 'nicht plausibel': 9
Disagreements:       8 (aus Evaluation ausgeschlossen)


## 4. LLM-Performance: Gesamtmetriken

In [7]:
# Das Sample enthaelt zwei Fallarten, die getrennt bewertet werden muessen:
#  (a) Extrahierte Relationen (Primaertyp PR-BER/PR-FLG/PR-INK):
#      "plausibel"       -> Extraktion bestaetigt (TP)
#      "nicht plausibel" -> faelschlich extrahiert (FP)
#  (b) Nicht-Extraktionen (PR-UNA):
#      "plausibel"       -> Fehlen einer Relation bestaetigt (TN)
#      "nicht plausibel" -> UEBERSEHENE Relation (FN der Extraktion)
# Ein pauschales "Recall = 1.0 per Definition" ist damit NICHT haltbar, da die
# PR-UNA-Faelle gerade die Vollstaendigkeit (partiell) mitpruefen.

def primary_type(val):
    if pd.isna(val):
        return "(kein)"
    types = [t.strip() for t in str(val).split(",")]
    return types[0] if types else "(kein)"

df_agreed["_primary_typ"] = df_agreed[COL_RELTYP].apply(primary_type)

mask_ext = df_agreed["_primary_typ"].isin(["PR-BER", "PR-FLG", "PR-INK"])
mask_una = ~mask_ext

n_ext = int(mask_ext.sum())
tp = int((df_agreed.loc[mask_ext, "gold"] == "plausibel").sum())
fp = n_ext - tp
n_una = int(mask_una.sum())
tn = int((df_agreed.loc[mask_una, "gold"] == "plausibel").sum())
fn = n_una - tn  # von Annotator:innen erkannte, vom Modell uebersehene Relationen

gesamt_acc = (df_agreed["gold"] == "plausibel").mean()
prec_ext   = tp / n_ext if n_ext else float("nan")
acc_una    = tn / n_una if n_una else float("nan")
recall_est = tp / (tp + fn) if (tp + fn) else float("nan")

# Legacy-Kennzahl (Vorhersage konstant "plausibel" ueber ALLE Faelle) — nur zur
# Rueckverfolgbarkeit frueherer Berichte; als Guetemass eingeschraenkt
# aussagekraeftig, da sie Extraktions- und Nicht-Extraktionsfaelle vermischt.
precision_legacy = gesamt_acc
f1_legacy = (2 * precision_legacy * 1.0 / (precision_legacy + 1.0)
             if precision_legacy > 0 else 0.0)

print("=" * 60)
print("LLM-PERFORMANCE (gegen Goldstandard, getrennt nach Fallart)")
print("=" * 60)
print(f"Gesamtplausibilitaet (Accuracy):        {gesamt_acc:.3f}  ({gesamt_acc*100:.1f}%)")
print(f"")
print(f"(a) Extrahierte Relationen (N={n_ext}):")
print(f"    Extraktionspraezision:              {prec_ext:.3f}  (TP={tp}, FP={fp})")
print(f"(b) Nicht-Extraktionen PR-UNA (N={n_una}):")
print(f"    Korrektheit der Nicht-Extraktion:   {acc_una:.3f}  (TN={tn}, FN={fn})")
print(f"")
print(f"Recall der Relationsextraktion (Stichproben-Schaetzung TP/(TP+FN)): {recall_est:.3f}")
print(f"Legacy: binaerer F1 bei konstanter 'plausibel'-Vorhersage: {f1_legacy:.3f} "
      f"(eingeschraenkt aussagekraeftig)")


LLM-PERFORMANCE (gegen Goldstandard, getrennt nach Fallart)
Gesamtplausibilitaet (Accuracy):        0.907  (90.7%)

(a) Extrahierte Relationen (N=60):
    Extraktionspraezision:              1.000  (TP=60, FP=0)
(b) Nicht-Extraktionen PR-UNA (N=37):
    Korrektheit der Nicht-Extraktion:   0.757  (TN=28, FN=9)

Recall der Relationsextraktion (Stichproben-Schaetzung TP/(TP+FN)): 0.870
Legacy: binaerer F1 bei konstanter 'plausibel'-Vorhersage: 0.951 (eingeschraenkt aussagekraeftig)


## 5. Performance nach Relationstyp

In [8]:
# Relationstypen können mehrere Werte haben (z.B. "PR-INK, PR-UNA")
# Wir verwenden den primären Typ (den ersten in der Liste)

def primary_type(val):
    if pd.isna(val):
        return "(kein)"
    types = [t.strip() for t in str(val).split(",")]
    return types[0] if types else "(kein)"

df_agreed["_primary_typ"] = df_agreed[COL_RELTYP].apply(primary_type)

print("=" * 60)
print("PERFORMANCE NACH RELATIONSTYP")
print("=" * 60)
print(f"{'Typ':<12} {'N':>5} {'Plausibel':>10} {'Nicht pl.':>10} {'Accuracy':>10}")
print("-" * 50)

type_rows = []
for typ in sorted(df_agreed["_primary_typ"].unique()):
    mask = df_agreed["_primary_typ"] == typ
    n = mask.sum()
    n_pl = (df_agreed.loc[mask, "gold"] == "plausibel").sum()
    n_np = (df_agreed.loc[mask, "gold"] == "nicht plausibel").sum()
    acc = n_pl / n if n > 0 else 0
    print(f"{typ:<12} {n:>5} {n_pl:>10} {n_np:>10} {acc:>10.1%}")
    type_rows.append({
        'Relationstyp': typ, 'N': n,
        'Plausibel': n_pl, 'Nicht plausibel': n_np,
        'Accuracy': round(acc, 3)
    })

print("-" * 50)
total_n = len(df_agreed)
total_pl = (df_agreed["gold"] == "plausibel").sum()
total_np = (df_agreed["gold"] == "nicht plausibel").sum()
print(f"{'GESAMT':<12} {total_n:>5} {total_pl:>10} {total_np:>10} {total_pl/total_n:>10.1%}")

if type_rows:
    worst = min(type_rows, key=lambda x: x['Accuracy'])
    best = max(type_rows, key=lambda x: x['Accuracy'])
    print(f"\nBester Typ:     {best['Relationstyp']} ({best['Accuracy']:.1%})")
    print(f"Schlechtester:  {worst['Relationstyp']} ({worst['Accuracy']:.1%})")

PERFORMANCE NACH RELATIONSTYP
Typ              N  Plausibel  Nicht pl.   Accuracy
--------------------------------------------------
PR-BER          54         54          0     100.0%
PR-FLG           6          6          0     100.0%
PR-UNA          37         28          9      75.7%
--------------------------------------------------
GESAMT          97         88          9      90.7%

Bester Typ:     PR-BER (100.0%)
Schlechtester:  PR-UNA (75.7%)


## 6. Fehleranalyse: Nicht-plausible Relationen

In [9]:
df_errors = df_agreed[df_agreed["gold"] == "nicht plausibel"].copy()

print("=" * 60)
print(f"FEHLERANALYSE: {len(df_errors)} NICHT-PLAUSIBLE RELATIONEN")
print("=" * 60)

if len(df_errors) > 0:
    print("\nVerteilung nach Relationstyp:")
    df_errors["_primary_typ"] = df_errors[COL_RELTYP].apply(primary_type)
    for typ, count in Counter(df_errors["_primary_typ"]).most_common():
        total_of_typ = (df_agreed["_primary_typ"] == typ).sum()
        print(f"  {typ}: {count} von {total_of_typ} ({count/total_of_typ*100:.1f}% Fehlerrate)")

    print(f"\nAlle nicht-plausiblen Relationen:")
    for idx, row in df_errors.iterrows():
        prop = row.get('Proposition', '')[:80] if 'Proposition' in df_errors.columns else ''
        rel = row.get(COL_INTRA_REL, '') if COL_INTRA_REL in df_errors.columns else ''
        typ = row.get(COL_RELTYP, '')
        print(f"  {idx}: {typ} | {rel}")
        if prop:
            print(f"    Proposition: {prop}")
else:
    print("\nKeine Fehler!")

FEHLERANALYSE: 9 NICHT-PLAUSIBLE RELATIONEN

Verteilung nach Relationstyp:
  PR-UNA: 9 von 37 (24.3% Fehlerrate)

Alle nicht-plausiblen Relationen:
  P(001.3): PR-UNA | nan
    Proposition: Die Vorbemerkungen sollen verhindern, dass der Eindruck erweckt wird, die univer
  P(022.3): PR-UNA | nan
    Proposition: Wahrnehmbare Grenzen, Erfahrungen des Fremden, Neuen und Ungewohnten bildeten ei
  P(022.4): PR-UNA | nan
    Proposition: Wahrnehmbare Grenzen, Erfahrungen des Fremden, Neuen und Ungewohnten bildeten di
  P(022.5): PR-UNA | nan
    Proposition: Wahrnehmbare Grenzen, Erfahrungen des Fremden, Neuen und Ungewohnten bildeten di
  P(025.3): PR-UNA | nan
    Proposition: Das Fränkisch-Alemannische wagt sich um 1170 als Literatursprache nur sehr vorsi
  P(038.3): PR-UNA | nan
    Proposition: Karl wird als »der Vater Europas« bezeichnet.
  P(038.4): PR-UNA | nan
    Proposition: Die Idee des Vaters Europas spiegelt sich heute in der jährlichen Verleihung des
  P(039.2): PR-UNA | nan
 

## 7. Disagreement-Analyse

In [10]:
print("=" * 60)
print(f"DISAGREEMENT-ANALYSE: {len(df_disagree)} F\u00c4LLE")
print("=" * 60)

if len(df_disagree) > 0:
    a1_pl_a2_np = ((df_disagree["A1_urteil"] == "plausibel") & (df_disagree["A2_urteil"] == "nicht plausibel")).sum()
    a1_np_a2_pl = ((df_disagree["A1_urteil"] == "nicht plausibel") & (df_disagree["A2_urteil"] == "plausibel")).sum()
    print(f"{name1} sagt plausibel, {name2} sagt nicht plausibel:   {a1_pl_a2_np}")
    print(f"{name1} sagt nicht plausibel, {name2} sagt plausibel:   {a1_np_a2_pl}")

    df_disagree["_primary_typ"] = df_disagree[COL_RELTYP].apply(primary_type)
    print(f"\nDisagreements nach Relationstyp:")
    for typ, count in Counter(df_disagree["_primary_typ"]).most_common():
        print(f"  {typ}: {count}")

    print(f"\nAlle Disagreements:")
    for idx, row in df_disagree.iterrows():
        typ = row.get(COL_RELTYP, '')
        rel = row.get(COL_INTRA_REL, '') if COL_INTRA_REL in df_disagree.columns else ''
        print(f"  {idx}: {typ} | {name1}={row['A1_urteil']}, {name2}={row['A2_urteil']}")
        if rel:
            print(f"    Relation: {rel}")
else:
    print("Keine Disagreements!")

DISAGREEMENT-ANALYSE: 8 FÄLLE
Annotator_1 sagt plausibel, Annotator_2 sagt nicht plausibel:   0
Annotator_1 sagt nicht plausibel, Annotator_2 sagt plausibel:   8

Disagreements nach Relationstyp:
  PR-UNA: 8

Alle Disagreements:
  P(005.1): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(008.3): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(008.4): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(020.8): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(020.9): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(025.1): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(025.2): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan
  P(039.1): PR-UNA | Annotator_1=nicht plausibel, Annotator_2=plausibel
    Relation: nan


## 8. Export

In [11]:
from openpyxl.styles import Font, PatternFill, Alignment
import re as _re

# Stichprobenumfang und -herkunft dynamisch dokumentieren
_nums = [int(m.group(1)) for m in
         (_re.match(r"P\((\d+)\.", str(i)) for i in df_agreed.index) if m]
_scope = f"Saetze {min(_nums)}-{max(_nums)}" if _nums else "unbekannt"
if "Kapitel" in df_agreed.columns:
    _kap = ", ".join(sorted(set(df_agreed["Kapitel"].dropna().astype(str))))
    _scope += f" | Kapitel: {_kap}"
_typen = sorted(t for t in df_agreed["_primary_typ"].unique())
_fehlend = [t for t in ["PR-BER", "PR-FLG", "PR-INK", "PR-UNA"] if t not in _typen]

summary = pd.DataFrame([
    {"Metrik": "Inter-Annotator Agreement", "Wert": "", "Detail": ""},
    {"Metrik": "  Cohen's Kappa", "Wert": f"{kappa:.3f}", "Detail": interpret_kappa(kappa)},
    {"Metrik": "  %-\u00dcbereinstimmung", "Wert": f"{agreement*100:.1f}%", "Detail": f"N={len(y1)}"},
    {"Metrik": "  Beide 'plausibel'", "Wert": str(both_pl), "Detail": f"{both_pl/len(y1)*100:.1f}%"},
    {"Metrik": "  Beide 'nicht plausibel'", "Wert": str(both_np), "Detail": f"{both_np/len(y1)*100:.1f}%"},
    {"Metrik": "  Disagreements", "Wert": str(disagree), "Detail": f"{disagree/len(y1)*100:.1f}%"},
    {"Metrik": "", "Wert": "", "Detail": ""},
    {"Metrik": "Goldstandard", "Wert": "", "Detail": ""},
    {"Metrik": "  Agreed Annotations", "Wert": str(len(df_agreed)), "Detail": f"{len(df_agreed)/len(df1)*100:.1f}%"},
    {"Metrik": "", "Wert": "", "Detail": ""},
    {"Metrik": "LLM-Performance", "Wert": "", "Detail": ""},
    {"Metrik": "  Gesamtplausibilit\u00e4t (Accuracy)", "Wert": f"{gesamt_acc:.3f}", "Detail": f"{gesamt_acc*100:.1f}%"},
    {"Metrik": "  Extraktionspr\u00e4zision (PR-BER/FLG/INK)", "Wert": f"{prec_ext:.3f}", "Detail": f"TP={tp}, FP={fp}, N={n_ext}"},
    {"Metrik": "  Korrektheit Nicht-Extraktion (PR-UNA)", "Wert": f"{acc_una:.3f}", "Detail": f"TN={tn}, FN={fn} (\u00fcbersehene Relationen), N={n_una}"},
    {"Metrik": "  Recall der Extraktion (Stichproben-Sch\u00e4tzung)", "Wert": f"{recall_est:.3f}", "Detail": "TP/(TP+FN); nur auf Basis der Stichprobe"},
    {"Metrik": "  F1 (Legacy, konstante 'plausibel'-Vorhersage)", "Wert": f"{f1_legacy:.3f}", "Detail": "eingeschr\u00e4nkt aussagekr\u00e4ftig, s. Methodische Hinweise"},
    {"Metrik": "", "Wert": "", "Detail": ""},
    {"Metrik": "Stichprobe", "Wert": "", "Detail": ""},
    {"Metrik": "  Herkunft", "Wert": str(len(df_agreed)), "Detail": _scope},
    {"Metrik": "  Typabdeckung", "Wert": ", ".join(_typen),
     "Detail": ("fehlend: " + ", ".join(_fehlend)) if _fehlend else "alle Typen vertreten"},
])

df_types = pd.DataFrame(type_rows)

err_cols = [c for c in ['Proposition', COL_RELTYP, COL_INTRA_REL, 'gold'] if c in df_errors.columns]
df_err_export = df_errors[err_cols].copy() if len(df_errors) > 0 else pd.DataFrame()

dis_cols = [c for c in ['Proposition', COL_RELTYP, COL_INTRA_REL, 'A1_urteil', 'A2_urteil'] if c in df_disagree.columns]
df_dis_export = df_disagree[dis_cols].copy() if len(df_disagree) > 0 else pd.DataFrame()

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    summary.to_excel(writer, index=False, sheet_name='Zusammenfassung')
    df_types.to_excel(writer, index=False, sheet_name='Performance pro Typ')
    if len(df_err_export) > 0:
        df_err_export.to_excel(writer, sheet_name='Nicht-plausible Relationen')
    if len(df_dis_export) > 0:
        df_dis_export.to_excel(writer, sheet_name='Disagreements')

    hf = PatternFill(start_color='2B4C7E', end_color='2B4C7E', fill_type='solid')
    for ws in writer.sheets.values():
        for cell in ws[1]:
            cell.fill = hf
            cell.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
            cell.alignment = Alignment(wrap_text=True, vertical='top')
        ws.auto_filter.ref = ws.dimensions

print(f"Exportiert: {OUTPUT_FILE}")


Exportiert: Beutin_complete_INF+System_EVAL_V1_eval_ergebnisse.xlsx


## Methodische Hinweise

**Zwei Fallarten, getrennte Kennzahlen.** Die Stichprobe enthält sowohl extrahierte
Relationen (PR-BER/PR-FLG/PR-INK) als auch Nicht-Extraktionen (PR-UNA). Ein
„nicht plausibel" bedeutet bei ersteren eine *fälschlich extrahierte* Relation
(False Positive), bei letzteren eine *übersehene* Relation (False Negative).
Die früher berichtete Formel „Recall = 1.0 per Definition" ist mit diesem
Stichprobendesign nicht vereinbar und wurde ersetzt durch:

- **Extraktionspräzision** — Anteil plausibler unter den extrahierten Relationen,
- **Korrektheit der Nicht-Extraktion** — Anteil bestätigter PR-UNA-Fälle,
- **Recall-Schätzung** TP/(TP+FN) auf Stichprobenbasis.

Der Legacy-F1 (konstante „plausibel"-Vorhersage über alle Fälle) wird nur zur
Rückverfolgbarkeit früherer Berichte mit ausgewiesen.

**Reichweite der Stichprobe.** Herkunft und Typabdeckung der Stichprobe werden im
Blatt *Zusammenfassung* dokumentiert. Stammen alle Fälle aus einem einzigen
Kapitel oder fehlen Relationstypen (z. B. PR-INK), sind die Kennzahlen nicht
ohne Weiteres auf das Gesamtkorpus übertragbar; für belastbare Aussagen ist eine
über Kapitel und Relationstypen geschichtete Zufallsstichprobe erforderlich.

**Goldstandard.** Nur übereinstimmende Urteile beider Annotator:innen gehen in den
Goldstandard ein; Disagreements werden gesondert ausgewiesen.
